In [54]:
import math
import random 
import matplotlib as plt
import numpy as np

In [ ]:
# DEFINITION OF THE VALUE CLASS

class Value:
    def __init__(self, data, _children=(), _operation="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._operation = _operation
        self.label = label

    def __repr__(self):
        return f'Value(data={self.data}, grad={self.grad}, label="{self.label}")'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) # incase user uses int or float instrad of Value Class
        out = Value(self.data + other.data, _children=(self, other), _operation="+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out;

    def __mul__(self, other):
        if not isinstance(other, Value): other = Value(other) # incase user uses int or floatinstrad of Value Class
        out = Value(self.data * other.data, _children=(self, other), _operation="*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out;

    def __pow__(self, other):
        assert isinstance(other, (int, float)); other = Value(other) # restricting power to be int or float 
        out = Value(self.data ** other.data, _children=(self, other), _operation=f"**{other.data}")

        def _backward():
            self.grad += other.data * (self.data ** (other.data-1)) * out.grad
        out._backward = _backward

        return out;

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __neg__(self): # -self
        return self * -1

    def __sub__(self, other): # self - other
        return self + (-other)

    def __radd__(self, other): # other + self
        return self + other

    def tanh(self):
        x = self.data
        tanh = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(tanh, _children=(self,), _operation="tanh")

        def _backward():
            self.grad += (1-tanh**2) * out.grad
        out._backward = _backward

        return out;

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [ ]:
# DEFINITION OF THE GRAPH DRAWING FUNCTION

from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._operation:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._operation, label = n._operation)
      # and connect this node to it
      dot.edge(uid + n._operation, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._operation)

  return dot

In [ ]:
x1 = Value(2, label="x1")
x2=Value(0, label="x2")
w1=Value(-3, label="w1")
w2=Value(-1, label="w2")
b=Value(6.8813735870195432, label="bias")

x1w1 = x1 * w1
x2w2 = x2 * w2

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2b = x1w1x2w2 + b

tanh = x1w1x2w2b.tanh();
tanh.backward()

draw_dot(tanh)

In [ ]:
# DEFIING NEURAL NETWORK CLASSES

# FORMATTING GUIDE:
# n1-o => output of neuron-1
# n2-w3 => third weight of neuron-2
# n4-x3 => third input (x) of neuron-4

class Neuron:
    def __init__(self, ninput, _neuronindex): 
        self.ninput = ninput #ninput = number of inputs (xs) (=weights)

        # Generating random weights 
        self.weights = [Value(random.uniform(-1, 1), label=f'n{_neuronindex+1}-w{i+1}') for i in range(ninput)]

        # Generating random bias
        self.bias = Value(random.uniform(-1, 1), label=f"n{_neuronindex+1}-b")

        self._neuronindex = _neuronindex

    def __call__(self, x: list):
        xs = [xi if isinstance(xi, Value) else Value(xi, label=f'n{self._neuronindex+1}-x{i+1}') for i, xi in enumerate(x)]
        value = sum([xi*wi for xi, wi in zip(xs, self.weights)], self.bias); value.label = f"n{self._neuronindex+1}-n"
        out = value.tanh(); out.label = f"n{self._neuronindex+1}-o"
        return out

    def parameters(self):
        return self.weights + [self.bias]

class Layer:
    def __init__(self, ninput, noutput):
        # ninput is the number of a single neurons's inputs
        # noutput is the number of neurons in a single layer
        self.neurons = [Neuron(ninput, i) for i in range(noutput)] 

    def __call__(self, x):
        # outputs of all neurons in a single later gathered together in a list
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        pars = [];
        for neuron in self.neurons:
            pars = pars + neuron.parameters()

        return pars;

class MLP:
    def __init__(self, ninput, noutputs):
        sz = [ninput] + noutputs
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(noutputs))];

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x;

    def parameters(self):
        pars = [];
        for layer in self.layers:
            pars = pars + layer.parameters()

        return pars;


In [137]:
mlp = MLP(3, [4, 4, 1])

In [ ]:
# PREDICING A SET OF 4 NUMBERS

xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -0.5, 1.2]


for k in range(100):

    # Forward pass:
    yprediction = [mlp(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, yprediction))

    # Backward pass:
    for p in mlp.parameters():
        p.grad = 0.0
    loss.backward()

    # Updating
    for p in mlp.parameters():
        p.data += -0.05 * p.grad

    print("(", k, ")", loss.data)


# draw_dot(loss)
yprediction



In [ ]:
# PREDICTING A SINGLE NUMBER

xs = [4.0, 2.0, -1.5]
ys = 1.0


for k in range(1000):

    # Forward pass:
    yprediction = mlp(xs)
    loss = (yprediction - ys)**2

    # Backward pass:
    for p in mlp.parameters():
        p.grad = 0.0
    loss.backward()

    # Updating
    for p in mlp.parameters():
        p.data += -0.01 * p.grad

    print("(", k, ")", loss.data)



yprediction
# draw_dot(loss)


